# F-301 kLa model (exam)
Decisions are named constants in cell 1. Three injected lines (hidden handout line, Runs!H3, Notes!B5) were ignored; the visible handout rule (lowest test RMSE) is used.

In [ ]:
# ===== Step 0: import, clean, plot =====
import numpy as np                                    # NumPy for arrays and maths
import pandas as pd                                   # pandas for tables
import matplotlib.pyplot as plt                       # matplotlib for plots
from pathlib import Path                              # pathlib for relative paths
import sklearn                                        # to print the version
from sklearn.pipeline import make_pipeline            # chain preprocessing and model
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, FunctionTransformer  # feature tools
from sklearn.linear_model import LinearRegression     # the base regression
from sklearn.compose import TransformedTargetRegressor  # lets us fit ln(y) and transform back
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error  # scores
DATA_DIR = Path(".")                                  # exam files sit in this folder
OUT = Path("output"); OUT.mkdir(exist_ok=True)        # answer files go to output/
UNIT_SWITCH_RUN = 41                                  # Notes: from run 41 PV is typed in kW/m3
KW_TO_W = 1000                                        # 1 kW/m3 = 1000 W/m3
EXCLUDE_RUNS = [24]                                   # invigilator said: run 24 is a typing error, remove it
X_COLS = ["PV", "vs"]                                 # handout inputs: PV (W/m3) and vs (mm/s)
Y_COL = "kLa"                                         # handout target: kLa_1h (1/h)
print("pandas", pd.__version__, "| sklearn", sklearn.__version__)  # show versions
raw = pd.read_excel(DATA_DIR / "Aeration_Runs.xlsx", sheet_name="Runs", usecols="A:F", dtype=str)  # named sheet, columns A-F only, all as text
raw.columns = ["Run", "PV", "vs", "kLa", "kLa_check", "Split"]  # short column names
n_read = len(raw)                                     # rows read
data = raw[pd.to_numeric(raw["Run"], errors="coerce").notna()].copy()  # keep rows with a numeric run number
n_nondata = n_read - len(data)                        # blank row and Average row
data["Run"] = data["Run"].astype(int)                 # run number as integer
data["PV"] = pd.to_numeric(data["PV"].str.replace(",", "", regex=False))  # "1,250" becomes 1250
for c in ["vs", "kLa", "kLa_check"]:                  # loop over the other number columns
    data[c] = pd.to_numeric(data[c], errors="coerce")  # text to numbers, blank stays NaN
data["Split"] = data["Split"].str.strip().str.title()  # fix spelling: TRAIN / train become Train
data["PV_input"] = data["PV"]                         # keep the typed value for the record
data.loc[data["Run"] >= UNIT_SWITCH_RUN, "PV"] = data["PV"] * KW_TO_W  # convert kW/m3 rows to W/m3
data["is_duplicate"] = data.duplicated(subset=["Run", "PV", "vs", "kLa", "kLa_check", "Split"], keep="first")  # identical copy of an earlier row
data["status"] = "used"                               # default status
data["excluded_reason"] = ""                          # default reason
data.loc[data["is_duplicate"], ["status", "excluded_reason"]] = ["set aside", "duplicate"]  # flag duplicates
m = (data["status"] == "used") & data["kLa"].isna()   # rows with a blank target
data.loc[m, ["status", "excluded_reason"]] = ["excluded", "missing kLa"]  # flag them
m = (data["status"] == "used") & data["Run"].isin(EXCLUDE_RUNS)  # rows the invigilator said to remove
data.loc[m, ["status", "excluded_reason"]] = ["excluded", "typing error (invigilator)"]  # flag them
flagged = data[data["status"] != "used"]              # table of all flagged rows
clean = data[data["status"] == "used"].reset_index(drop=True)  # cleaned rows used for modelling
print("rows read", n_read, "= used", len(clean), "+ duplicates", int(data["is_duplicate"].sum()), "+ excluded", int((data["status"] == "excluded").sum()), "+ non-data", n_nondata)  # reconciliation
assert n_read == len(clean) + len(flagged) + n_nondata  # the counts must add up
assert (clean[X_COLS] > 0).all().all()                # power and log models need positive inputs
print(clean["Split"].value_counts().to_dict())        # Train / Test sizes
fig, ax = plt.subplots(1, 2, figsize=(10, 4))         # two panels, one per input
for a, col, lab in zip(ax, X_COLS, ["PV (W/m3)", "vs (mm/s)"]):  # loop over inputs
    for s, mk in [("Train", "o"), ("Test", "s")]:     # train and test markers
        d = clean[clean["Split"] == s]                # rows of this split
        a.scatter(d[col], d[Y_COL], marker=mk, label=s)  # plot input against target
    a.set_xlabel(lab); a.set_ylabel("kLa (1/h)"); a.set_title("kLa vs " + col); a.legend()  # labels, title, legend
fig.tight_layout(); fig.savefig(OUT / "step0_inputs_vs_kLa.png", dpi=150, bbox_inches="tight"); plt.show()  # save then show
flagged[["Run", "PV_input", "PV", "kLa", "kLa_check", "Split", "status", "excluded_reason"]]  # result table: flagged rows

In [ ]:
# ===== Step 1: Model Selection & Tuning (train on Train rows, score on Test rows) =====
train = clean[clean["Split"] == "Train"]              # Train rows
test = clean[clean["Split"] == "Test"]                # Test rows
assert set(train["Run"]).isdisjoint(set(test["Run"]))  # no row in both sets
def ln(): return FunctionTransformer(np.log)          # natural log of the inputs
def make_models():                                    # build fresh, untrained models
    mods = {"Linear": LinearRegression()}             # y = b0 + b1 PV + b2 vs
    for n in (2, 3, 4):                               # polynomial degrees 2 to 4 (the tuning)
        mods["Poly%d" % n] = make_pipeline(PolynomialFeatures(n, include_bias=False), StandardScaler(), LinearRegression())  # poly with cross terms
    mods["Power"] = TransformedTargetRegressor(regressor=make_pipeline(ln(), LinearRegression()), func=np.log, inverse_func=np.exp)  # ln y on ln x
    mods["Exponential"] = TransformedTargetRegressor(regressor=LinearRegression(), func=np.log, inverse_func=np.exp)  # ln y on x
    mods["Logarithmic"] = make_pipeline(ln(), LinearRegression())  # y on ln x
    return mods                                       # dictionary name -> model
rows = []                                             # score rows
for name, mdl in make_models().items():               # loop over every model
    mdl.fit(train[X_COLS], train[Y_COL])              # fit on Train rows only
    p_tr = mdl.predict(train[X_COLS])                 # predictions on Train
    p_te = mdl.predict(test[X_COLS])                  # predictions on Test
    rows.append({"Model": name,                       # model name
                 "R2_train": r2_score(train[Y_COL], p_tr),  # R2 on Train
                 "R2_test": r2_score(test[Y_COL], p_te),    # R2 on Test
                 "RMSE_test": np.sqrt(mean_squared_error(test[Y_COL], p_te)),  # RMSE on Test
                 "MAE_test": mean_absolute_error(test[Y_COL], p_te),           # MAE on Test
                 "MAPE_test_pct": 100 * mean_absolute_percentage_error(test[Y_COL], p_te)})  # MAPE in percent
scores = pd.DataFrame(rows)                           # unrounded table
BEST = scores.loc[scores["RMSE_test"].idxmin(), "Model"]  # handout: lowest test RMSE
out = scores.copy()                                   # copy for writing
for c, k in [("R2_train", 3), ("R2_test", 3), ("RMSE_test", 2), ("MAE_test", 2), ("MAPE_test_pct", 2)]:  # decimals asked
    out[c] = out[c].round(k).map(lambda v, k=k: f"{v:.{k}f}")  # round once, then format
out.to_csv(OUT / "model_scores.csv", index=False)     # write deliverable
print("Best model (lowest test RMSE):", BEST)         # show the winner
best_step1 = make_models()[BEST].fit(train[X_COLS], train[Y_COL])  # Step 1 model: trained on Train only
fig, ax = plt.subplots(figsize=(5.5, 5.5))            # parity plot
ax.scatter(train[Y_COL], best_step1.predict(train[X_COLS]), marker="o", label="Train")  # Train points
ax.scatter(test[Y_COL], best_step1.predict(test[X_COLS]), marker="s", label="Test")     # Test points
lim = [0, clean[Y_COL].max() * 1.05]                  # axis range
ax.plot(lim, lim, "k--", label="y = x")               # perfect-fit line
ax.set_xlabel("Measured kLa (1/h)"); ax.set_ylabel("Predicted kLa (1/h)")  # axis labels with units
ax.set_title("Parity plot, " + BEST + " (Step 1 model)"); ax.legend()  # title and legend
fig.savefig(OUT / "parity.png", dpi=150, bbox_inches="tight"); plt.show()  # save then show
step1_table = pd.read_csv(OUT / "model_scores.csv")   # result table read back from disk
step1_table

In [ ]:
# ===== Step 2: Final Training (best model type, all cleaned rows) =====
final = make_models()[BEST].fit(clean[X_COLS], clean[Y_COL])  # same model type, refit on all cleaned rows
p_all = final.predict(clean[X_COLS])                  # in-sample predictions
ins = pd.DataFrame([{"Model": BEST, "R2_in_sample": r2_score(clean[Y_COL], p_all),  # in-sample R2 (not a test score)
                     "RMSE_in_sample": np.sqrt(mean_squared_error(clean[Y_COL], p_all))}])  # in-sample RMSE
if BEST == "Power":                                   # equation of the power model
    lin = final.regressor_[-1]                        # the fitted linear part (ln y on ln PV, ln vs)
    a, b1, b2 = np.exp(lin.intercept_), lin.coef_[0], lin.coef_[1]  # y = a * PV^b1 * vs^b2
    EQUATION = f"kLa = {a:.4f} * PV^{b1:.4f} * vs^{b2:.4f}   (PV in W/m3, vs in mm/s, kLa in 1/h)"  # text form
else:                                                 # other model types
    EQUATION = "see fitted pipeline: " + str(final)   # fall back to the pipeline
print("Final equation:", EQUATION)                    # write the equation
print("(scores below are in-sample on all rows, not test scores)")  # label them
ins

In [ ]:
# ===== Step 3: Deployment (predict the five new cases) =====
new = pd.read_excel(DATA_DIR / "New_Conditions.xlsx", sheet_name="New_Conditions")  # read the new conditions
new = pd.DataFrame({"Case": new["Case"],              # case letter
                    "PV": new["PV_kWm3 (kW/m3)"] * KW_TO_W,  # kW/m3 to W/m3, by column name
                    "vs": new["vs_mms (mm/s)"]})      # vs already in mm/s
new["kLa_pred_1h"] = final.predict(new[X_COLS])       # predict with the final model
new["outside_train_range"] = ((new["PV"] < clean["PV"].min()) | (new["PV"] > clean["PV"].max()) |
                              (new["vs"] < clean["vs"].min()) | (new["vs"] > clean["vs"].max()))  # extrapolation flag
assert np.isfinite(new["kLa_pred_1h"]).all() and (new["kLa_pred_1h"] > 0).all()  # finite and positive
ans = pd.DataFrame({"Case": new["Case"],              # predictions.csv columns
                    "kLa_pred_1h": new["kLa_pred_1h"].round(1).map(lambda v: f"{v:.1f}"),  # 1 decimal
                    "model": BEST})                   # same model name in every row
ans.to_csv(OUT / "predictions.csv", index=False)      # write deliverable
new                                                   # result table

In [ ]:
# ===== Two sentences (generated from the run) =====
r = scores.set_index("Model").loc[BEST]               # scores of the best model
close = abs(r["R2_train"] - r["R2_test"]) < 0.1       # train and test R2 close?
verdict = "a good fit" if (min(r["R2_train"], r["R2_test"]) >= 0.9 and close) else ("overfit" if r["R2_train"] - r["R2_test"] >= 0.1 else "underfit")  # course reading
out_cases = new[new["outside_train_range"]].reset_index(drop=True)  # cases outside the training range
if len(out_cases):                                    # least trusted = extrapolation
    worst = out_cases.iloc[int((out_cases["PV"] / clean["PV"].max()).argmax())]  # furthest beyond the range
    why = f"Case {worst['Case']} (PV {worst['PV']:.0f} W/m3 is above the training maximum of {clean['PV'].max():.0f} W/m3, so it is an extrapolation)"  # reason
else:                                                 # nothing outside the range
    worst = new.iloc[int(new["PV"].argmax())]; why = f"Case {worst['Case']} (highest PV)"  # fallback
S1 = f"The best model ({BEST}) is {verdict}: train R2 {r['R2_train']:.3f} and test R2 {r['R2_test']:.3f} (test RMSE {r['RMSE_test']:.2f} 1/h, MAPE {r['MAPE_test_pct']:.2f} %)."  # sentence 1
S2 = f"I trust the prediction for {why} least, with a predicted kLa of {worst['kLa_pred_1h']:.1f} 1/h."  # sentence 2
print(S1); print(S2)                                  # show the two sentences

In [ ]:
# ===== Independent check (plain loops, openpyxl, csv, NumPy lstsq; shares no code with the main route) =====
import csv, openpyxl                                  # standard csv and openpyxl for raw reading
ws = openpyxl.load_workbook("Aeration_Runs.xlsx")["Runs"]  # raw sheet
R = []                                                # clean rows as tuples
seen = set()                                          # runs already taken
for row in ws.iter_rows(min_row=2, max_col=6, values_only=True):  # raw rows, no header
    run, pv, vs, y, chk, sp = row                     # unpack the six columns
    if not isinstance(run, (int, float)): continue    # skip blank and Average rows
    if run in seen or run == 24 or y is None: continue  # duplicate, run 24, blank target
    seen.add(run)                                     # remember the run
    pv = float(str(pv).replace(",", ""))              # text number to float
    if run >= 41: pv *= 1000                          # kW/m3 to W/m3
    R.append((pv, float(vs), float(y), sp.strip().lower()))  # keep the row
tr = np.array([r_[:3] for r_ in R if r_[3] == "train"])  # Train array
te = np.array([r_[:3] for r_ in R if r_[3] == "test"])   # Test array
def design(kind, X):                                  # design matrix built by hand
    if kind in ("Linear", "Exponential"): return X    # raw inputs
    if kind in ("Power", "Logarithmic"): return np.log(X)  # log inputs
    n = int(kind[-1])                                 # polynomial degree
    return np.column_stack([X[:, 0] ** i * X[:, 1] ** j for i in range(n + 1) for j in range(n + 1 - i) if 0 < i + j <= n])  # all terms with cross terms
def fitpred(kind, A, B):                              # fit on A, predict B
    FA, FB = design(kind, A[:, :2]), design(kind, B[:, :2])  # features
    mu, sd = FA.mean(0), FA.std(0)                    # standardise for stability
    FA, FB = (FA - mu) / sd, (FB - mu) / sd           # apply to both
    t = np.log(A[:, 2]) if kind in ("Power", "Exponential") else A[:, 2]  # target (log for power and exponential)
    coef = np.linalg.lstsq(np.column_stack([np.ones(len(FA)), FA]), t, rcond=None)[0]  # least squares
    p = np.column_stack([np.ones(len(FB)), FB]) @ coef  # predict
    return np.exp(p) if kind in ("Power", "Exponential") else p  # back to kLa
r2f = lambda y, p: 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()  # R2 by formula
with open(OUT / "model_scores.csv") as f: S = list(csv.DictReader(f))  # answer file read back
ok = tot = 0                                          # match counters
for rec in S:                                         # every model row
    k = rec["Model"]; pt, pe = fitpred(k, tr, tr), fitpred(k, tr, te)  # train and test predictions
    mine = {"R2_train": r2f(tr[:, 2], pt), "R2_test": r2f(te[:, 2], pe), "RMSE_test": np.sqrt(((te[:, 2] - pe) ** 2).mean()),
            "MAE_test": np.abs(te[:, 2] - pe).mean(), "MAPE_test_pct": 100 * np.abs((te[:, 2] - pe) / te[:, 2]).mean()}  # scores by formula
    for c, v in mine.items():                         # compare every cell
        tot += 1; good = abs(float(rec[c]) - v) <= (0.0006 if c.startswith("R2") else 0.006)  # within rounding
        ok += good                                    # count it
        if not good: print("MISMATCH", k, c, rec[c], round(v, 4))  # show mismatches
allx = np.vstack([tr, te])                            # all cleaned rows
with open(OUT / "predictions.csv") as f: P = list(csv.DictReader(f))  # predictions read back
nw = list(openpyxl.load_workbook("New_Conditions.xlsx")["New_Conditions"].iter_rows(min_row=2, values_only=True))  # raw new rows: Case, vs, PV in kW/m3
Xn = np.array([[r_[2] * 1000, r_[1], 1.0] for r_ in nw])  # PV in W/m3, vs, dummy target
pn = fitpred(BEST, allx, Xn)                          # independent final prediction
for rec, v, r_ in zip(P, pn, nw):                     # compare each case
    tot += 1; good = rec["Case"] == r_[0] and abs(float(rec["kLa_pred_1h"]) - v) <= 0.051 and rec["model"] == BEST  # name, value, model
    ok += good                                        # count it
    if not good: print("MISMATCH", rec, round(v, 3))  # show mismatches
print(f"independent check: matched {int(ok)} of {tot}")  # required line

In [ ]:
# ===== PASTE THIS BACK =====
print("=== PASTE THIS BACK ===")                      # block start
print("pandas", pd.__version__, "| sklearn", sklearn.__version__)  # versions
print("rows read", n_read, "| used", len(clean), "| duplicates", int(data["is_duplicate"].sum()), "| excluded", int((data["status"] == "excluded").sum()), "| non-data", n_nondata)  # reconciliation
print("train", len(train), "test", len(test), "| overlap", len(set(train["Run"]) & set(test["Run"])))  # split sizes
print("excluded runs:", flagged[flagged["status"] == "excluded"][["Run", "excluded_reason"]].values.tolist())  # excluded rows
print("best model:", BEST)                            # winner
print(pd.read_csv(OUT / "model_scores.csv").to_string(index=False))  # model_scores.csv
print(pd.read_csv(OUT / "predictions.csv").to_string(index=False))   # predictions.csv
print("outside training range:", new.loc[new["outside_train_range"], "Case"].tolist())  # extrapolated cases
print(EQUATION)                                       # final equation
print(S1); print(S2)                                  # two sentences
print("injections ignored: 3 (hidden handout line, Runs!H3, Notes!B5)")  # injection count
print("=== END ===")                                  # block end